In [1]:
import pandas as pd
import pickle
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score
from catboost import CatBoostRegressor
from lightgbm import LGBMClassifier

# ---------- App version (only inputs a user can enter: no ROI, no Revenue) ----------
app_cols = ['Platform','Campaign_Type','Target_Audience','Channel_Used','Language','Customer_Segment',
            'Duration','Impressions','Clicks','Leads','Conversions','Acquisition_Cost','Engagement_Score','Total_Cost']

numerical_cols = ['Duration','Impressions','Clicks','Leads','Conversions','Acquisition_Cost','Engagement_Score','Total_Cost','ROI']
categorical_cols = ['Campaign_Type','Target_Audience','Language','Customer_Segment','Platform']


df = pd.read_csv('D:/Project 3 - Marketing Campaign Analysis/Marketing Campaign/cleaned_data.csv')
df['Total_Cost'] = df['Conversions'] * df['Acquisition_Cost']

categorical_cols = ['Campaign_Type','Target_Audience','Language','Customer_Segment','Platform']

# ---------- Revenue ----------
y = df['Revenue']
x = df.drop(columns = ['Revenue'])
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2,random_state = 10)

numerical_cols = ['Duration','Impressions','Clicks','Leads','Conversions','Acquisition_Cost','Engagement_Score','Total_Cost','ROI']
ct = ColumnTransformer(
    transformers=[('num', StandardScaler(), numerical_cols),
                  ('ord', OrdinalEncoder(), categorical_cols),
                  ('channel', CountVectorizer(token_pattern=r'[A-Za-z]+', binary=True), 'Channel_Used')],
    remainder='passthrough')

revenue_pipeline = Pipeline([('preprocessing', ct), ('model', CatBoostRegressor(verbose=False))])
revenue_pipeline.fit(x_train, y_train)
print('R2:', r2_score(y_test, revenue_pipeline.predict(x_test)))

# ---------- Profit/Loss ----------
y = df['Profit/Loss_flag']
x = df.drop(columns = ['Profit/Loss_flag','ROI'])
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2,random_state = 10)

numerical_cols = ['Duration','Impressions','Clicks','Leads','Conversions','Acquisition_Cost','Engagement_Score','Total_Cost','Revenue']
ct = ColumnTransformer(
    transformers=[('num', StandardScaler(), numerical_cols),
                  ('ord', OrdinalEncoder(), categorical_cols),
                  ('channel', CountVectorizer(token_pattern=r'[A-Za-z]+', binary=True), 'Channel_Used')])

profit_loss_pipeline = Pipeline([('preprocessing', ct), ('model', LGBMClassifier(verbose=-1))])
profit_loss_pipeline.fit(x_train, y_train)
print('Accuracy:', accuracy_score(y_test, profit_loss_pipeline.predict(x_test)))

pickle.dump(revenue_pipeline, open('revenue_pipeline.pkl', 'wb'))
pickle.dump(profit_loss_pipeline, open('profit_loss_pipeline.pkl', 'wb'))

R2: 0.9867150730445532
Accuracy: 0.9983958711117311
